# MODÈLE — `<source>` : `<table>` landing → bronze (COPY INTO)

**Comment l'utiliser** (manuel 6, recette B) :
1. Importer dans `ingestion/<source>/process/03_bronze/01_loaders/`, renommer `<source>_<table>_b_loader`.
2. Chercher **`À ADAPTER`** : `SOURCE`, `TABLE`, `CLE_METIER` (colonne qui identifie une ligne, pour compter les doublons).
3. Run all. Relancer une 2e fois : **0 ligne** doit être ajoutée (chaque fichier n'est lu qu'une fois).

Cible : `<CATALOG>.bronze.<source>_<table>` + 3 colonnes techniques `_date_depot`, `_fichier_source`, `_date_chargement_bronze`.

In [0]:
dbutils.widgets.text("CATALOG", "dev", "Catalog cible")

In [0]:
CATALOG = dbutils.widgets.get("CATALOG").strip()

# ---------- À ADAPTER ----------
SOURCE     = "open_meteo"
TABLE      = "meteo_journaliere"
CLE_METIER = ["_param_latitude", "_param_longitude", "time"]   # point géographique + jour
# --------------------------------

CHEMIN_LANDING = f"/Volumes/{CATALOG}/landing/files/{SOURCE}/{TABLE}/"
TABLE_BRONZE   = f"{CATALOG}.bronze.{SOURCE}_{TABLE}"
print("Source :", CHEMIN_LANDING, "| Cible :", TABLE_BRONZE)

## Création de la table (vide, sans colonnes : COPY INTO les déduira)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_BRONZE}
    COMMENT 'Bronze {SOURCE} {TABLE} : tous les dépôts de la landing + _date_depot, _fichier_source, _date_chargement_bronze'
""")
print("✅ Table prête :", TABLE_BRONZE)

## État avant chargement

In [0]:
import os
fichiers_landing = [os.path.join(d, f) for d, _, fs in os.walk(CHEMIN_LANDING) for f in fs if f.endswith(".parquet")]
try:
    lignes_avant = spark.table(TABLE_BRONZE).count()
except Exception as e:
    if "DELTA_READ_TABLE_WITHOUT_COLUMNS" in str(e):   # 1er lancement : table encore sans colonnes
        lignes_avant = 0
    else:
        raise
print(f"📂 Fichiers parquet en landing : {len(fichiers_landing)} | 📊 Lignes en bronze avant : {lignes_avant}")

## Chargement : COPY INTO (idempotent : un fichier déjà chargé est ignoré)

In [0]:
requete_copy = f"""
COPY INTO {TABLE_BRONZE}
FROM (
  SELECT *,
         timestamp_seconds(CAST(_dlt_load_id AS DOUBLE)) AS _date_depot,
         _metadata.file_path                             AS _fichier_source,
         current_timestamp()                             AS _date_chargement_bronze
  FROM '{CHEMIN_LANDING}'
)
FILEFORMAT = PARQUET
PATTERN = '*/*/*/*.parquet'
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
"""
display(spark.sql(requete_copy))

## Vérifications

In [0]:
from pyspark.sql import functions as F
df = spark.table(TABLE_BRONZE)
lignes_apres = df.count()
print(f"📊 Lignes : {lignes_avant} → {lignes_apres} (+{lignes_apres - lignes_avant}) | 🧱 Colonnes : {len(df.columns)}")
display(df.groupBy("_date_depot", "_fichier_source").agg(F.count("*").alias("nb_lignes")).orderBy("_date_depot"))
if CLE_METIER:
    distincts = df.select(CLE_METIER).distinct().count()
    print(f"🔁 Lignes : {lignes_apres} | {CLE_METIER} distincts : {distincts} | doublons (attendus si plusieurs dépôts) : {lignes_apres - distincts}")

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {TABLE_BRONZE}")
          .select("version", "timestamp", "userName", "operation", "operationMetrics").limit(5))